In [1]:
import pandas as pd
import joblib
import os

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
df = pd.read_csv("mess_food_dataset.csv")

In [4]:
# Input features and target
features = [
    "day_of_week",
    "meal_type",
    "menu",
    "is_weekend"
]

In [5]:
target = "amount_of_food_required_kg"

In [6]:
X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# Preprocess categorical and numerical inputs
categorical = ["day_of_week", "meal_type", "menu"]
numerical = [
    "is_weekend"
]

In [8]:
preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical),

    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numerical)
])

In [9]:

# Models
models = {
    "Linear_Regression": LinearRegression(),

    "Ridge": Ridge(alpha=10.0),

    "Lasso": Lasso(alpha=0.001, max_iter=20000),

    "KNN": KNeighborsRegressor(
        n_neighbors=7,
        weights="distance",
        p=2
    ),

    "SVR": SVR(
        kernel="rbf",
        C=100,
        gamma="scale",
        epsilon=0.1
    ),

    "Random_Forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient_Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=2,
        min_samples_split=4,
        min_samples_leaf=2,
        loss="huber",
        random_state=42
    )
}


In [10]:
os.makedirs("saved_models", exist_ok=True)
results = []

In [11]:
# Train, evaluate and save each model
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    # Accuracy-like score based on relative absolute error
    accuracy = (1 - mae / y_test.abs().mean()) * 100

    results.append({
        "Model": name,
        "MAE_kg": round(mae, 3),
        "R2": round(r2, 3),
        "Accuracy_%": round(accuracy, 2)
    })

    joblib.dump(
        pipeline,
        f"saved_models/{name}.joblib"
    )

    print(
        f"{name}: "
        f"Accuracy={accuracy:.2f}%, "
        f"MAE={mae:.3f} kg, "
        f"R2={r2:.3f}"
    )

Linear_Regression: Accuracy=92.49%, MAE=23.605 kg, R2=0.892
Ridge: Accuracy=92.49%, MAE=23.601 kg, R2=0.892
Lasso: Accuracy=92.49%, MAE=23.605 kg, R2=0.892
KNN: Accuracy=92.18%, MAE=24.577 kg, R2=0.881
SVR: Accuracy=92.34%, MAE=24.050 kg, R2=0.888
Random_Forest: Accuracy=92.46%, MAE=23.689 kg, R2=0.891
Gradient_Boosting: Accuracy=92.55%, MAE=23.391 kg, R2=0.894


In [13]:
# Compare model performance
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE_kg")

print("\nModel comparison:")
print(results_df.to_string(index=False))

results_df.to_csv("model_comparison.csv", index=False)
print("\nAll models saved in the saved_models folder.")


Model comparison:
            Model  MAE_kg    R2  Accuracy_%
Gradient_Boosting  23.391 0.894       92.55
            Ridge  23.601 0.892       92.49
Linear_Regression  23.605 0.892       92.49
            Lasso  23.605 0.892       92.49
    Random_Forest  23.689 0.891       92.46
              SVR  24.050 0.888       92.34
              KNN  24.577 0.881       92.18

All models saved in the saved_models folder.
